# Track A — Split & Verifikasi Dataloader (Fase 2)
**BDC Satria Data 2026 — Klasifikasi Citra Sampah**

Notebook ini mencakup:
- Fase 2: Stratified 5-Fold Split (+ group-aware jika ada duplikat)
- Hitung class weights
- Verifikasi PyTorch DataLoader

> **Prerequisite:** Jalankan `01_eda.ipynb` dulu → hasilkan `df_clean.csv` dan `eda_stats.json`
> **Output utama:** `folds.csv`, `class_weights.npy`

## 0. Setup

In [ ]:
!pip install -q torch torchvision scikit-learn pandas numpy matplotlib

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
import json
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.model_selection import StratifiedKFold, StratifiedGroupKFold

import torch
from torch.utils.data import Dataset, DataLoader
from PIL import Image
import torchvision.transforms as T

# ─── KONFIGURASI ──────────────────────────────────────────────────────────────
DATA_ROOT   = Path('/content/drive/MyDrive/BDC2026')
TEST_DIR    = DATA_ROOT / 'test'
SUB_CSV     = DATA_ROOT / 'submission.csv'
OUTPUT_DIR  = Path('/content/track_a_outputs')   # harus sama dengan notebook 01
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

RANDOM_SEED = 42
N_FOLDS     = 5
IMG_SIZE    = 224     # ← resolusi input (tetapkan setelah lihat hasil EDA)

CLASS_NAMES = ['Recyclable', 'Electronic', 'Organic']

print(f'OUTPUT_DIR: {OUTPUT_DIR}')
print(f'Seed: {RANDOM_SEED} | Folds: {N_FOLDS} | ImgSize: {IMG_SIZE}')

## 1. Load Hasil EDA

In [ ]:
df = pd.read_csv(OUTPUT_DIR / 'df_clean.csv')

with open(OUTPUT_DIR / 'eda_stats.json') as f:
    eda_stats = json.load(f)

print(f'df_clean loaded: {len(df)} gambar')
print(f'Kolom: {df.columns.tolist()}')
print(f'Need group-aware split: {eda_stats.get("need_group_aware_split", False)}')

## 2. Stratified 5-Fold Split

In [ ]:
df = df.copy()
df['fold'] = -1

need_group_split = eda_stats.get('need_group_aware_split', False)

if need_group_split and 'group_id' in df.columns:
    print('Menggunakan StratifiedGroupKFold (ada duplikat) ...')
    kf = StratifiedGroupKFold(n_splits=N_FOLDS, shuffle=True, random_state=RANDOM_SEED)
    for fold_idx, (_, val_idx) in enumerate(kf.split(df, df['label'], groups=df['group_id'])):
        df.loc[val_idx, 'fold'] = fold_idx
else:
    print('Menggunakan StratifiedKFold (tidak ada duplikat signifikan) ...')
    kf = StratifiedKFold(n_splits=N_FOLDS, shuffle=True, random_state=RANDOM_SEED)
    for fold_idx, (_, val_idx) in enumerate(kf.split(df, df['label'])):
        df.loc[val_idx, 'fold'] = fold_idx

assert (df['fold'] == -1).sum() == 0, 'Ada gambar yang tidak masuk fold!'
print(f'\nDistribusi fold:')
print(df.groupby(['fold', 'label']).size().unstack())

In [ ]:
# Verifikasi proporsi kelas per fold
fig, axes = plt.subplots(1, N_FOLDS, figsize=(15, 4), sharey=True)
for fold_idx in range(N_FOLDS):
    val_fold = df[df['fold'] == fold_idx]
    proportions = val_fold['label'].value_counts(normalize=True).sort_index()
    axes[fold_idx].bar(CLASS_NAMES, proportions.values,
                       color=['#2ecc71','#3498db','#e67e22'])
    axes[fold_idx].set_title(f'Fold {fold_idx}\n(n={len(val_fold)})')
    axes[fold_idx].set_ylim(0, 1)
    axes[fold_idx].tick_params(axis='x', rotation=30)

plt.suptitle('Proporsi Kelas per Fold (Validation Set)', fontweight='bold')
plt.tight_layout()
plt.savefig(OUTPUT_DIR / 'fold_proportions.png', dpi=150, bbox_inches='tight')
plt.show()

In [ ]:
# Verifikasi no-overlap antar fold
for fold_i in range(N_FOLDS):
    for fold_j in range(fold_i + 1, N_FOLDS):
        fp_i = set(df[df['fold'] == fold_i]['filepath'])
        fp_j = set(df[df['fold'] == fold_j]['filepath'])
        overlap = fp_i & fp_j
        assert len(overlap) == 0, f'LEAKAGE! Fold {fold_i} & {fold_j} share {len(overlap)} gambar'

print('No-overlap antar fold: LOLOS')

In [ ]:
# Simpan folds.csv — ARTEFAK UTAMA
folds_csv_path = OUTPUT_DIR / 'folds.csv'
df[['filepath', 'label', 'fold']].to_csv(folds_csv_path, index=False)
print(f'folds.csv disimpan ke: {folds_csv_path}')
print(df[['filepath', 'label', 'fold']].head())

# Catat seed
split_meta = {
    'random_seed': RANDOM_SEED,
    'n_folds': N_FOLDS,
    'method': 'StratifiedGroupKFold' if need_group_split else 'StratifiedKFold',
    'img_size': IMG_SIZE,
    'total_train': len(df),
}
with open(OUTPUT_DIR / 'split_metadata.json', 'w') as f:
    json.dump(split_meta, f, indent=2)
print('\nsplit_metadata.json disimpan')

## 3. Hitung Class Weights

In [ ]:
labels = df['label'].values
n_classes = 3
counts = np.bincount(labels, minlength=n_classes).astype(float)
class_weights = len(labels) / (n_classes * counts)

print('Class weights (untuk CrossEntropyLoss):')
for cls, w in zip(CLASS_NAMES, class_weights):
    print(f'  {cls}: {w:.4f}')

# Simpan
np.save(OUTPUT_DIR / 'class_weights.npy', class_weights)
print(f'\nclass_weights.npy disimpan')
print('Cara pakai di Track B:')
print('  weights = torch.tensor(np.load("class_weights.npy"), dtype=torch.float).to(device)')
print('  criterion = nn.CrossEntropyLoss(weight=weights)')

## 4. Verifikasi DataLoader

In [ ]:
# Copy paste class WasteDataset + transforms dari dataset.py
# (versi inline untuk Colab tanpa perlu import file)

IMAGENET_MEAN = [0.485, 0.456, 0.406]
IMAGENET_STD  = [0.229, 0.224, 0.225]

def get_train_transform(img_size=224):
    return T.Compose([
        T.RandomResizedCrop(img_size, scale=(0.7, 1.0)),
        T.RandomHorizontalFlip(),
        T.ColorJitter(brightness=0.3, contrast=0.3, saturation=0.2, hue=0.05),
        T.RandomRotation(15),
        T.ToTensor(),
        T.Normalize(mean=IMAGENET_MEAN, std=IMAGENET_STD),
    ])

def get_eval_transform(img_size=224):
    return T.Compose([
        T.Resize((img_size, img_size)),
        T.ToTensor(),
        T.Normalize(mean=IMAGENET_MEAN, std=IMAGENET_STD),
    ])

class WasteDataset(Dataset):
    def __init__(self, df, transform=None, is_test=False):
        self.df = df.reset_index(drop=True)
        self.transform = transform
        self.is_test = is_test
    def __len__(self):
        return len(self.df)
    def __getitem__(self, idx):
        row = self.df.iloc[idx]
        img = Image.open(row['filepath']).convert('RGB')
        if self.transform:
            img = self.transform(img)
        if self.is_test:
            return img, row['filepath']
        return img, int(row['label'])

print('WasteDataset defined')

In [ ]:
# Test fold 0
VAL_FOLD = 0
folds_df = pd.read_csv(folds_csv_path)
df_train = folds_df[folds_df['fold'] != VAL_FOLD].reset_index(drop=True)
df_val   = folds_df[folds_df['fold'] == VAL_FOLD].reset_index(drop=True)

train_ds = WasteDataset(df_train, transform=get_train_transform(IMG_SIZE))
val_ds   = WasteDataset(df_val,   transform=get_eval_transform(IMG_SIZE))

train_loader = DataLoader(train_ds, batch_size=16, shuffle=True,  num_workers=2)
val_loader   = DataLoader(val_ds,   batch_size=16, shuffle=False, num_workers=2)

# Ambil 1 batch
imgs, labels = next(iter(train_loader))
print(f'Train batch — images: {imgs.shape}, labels: {labels.tolist()}')
assert imgs.shape == (16, 3, IMG_SIZE, IMG_SIZE), f'Shape salah: {imgs.shape}'
assert all(l in {0, 1, 2} for l in labels.tolist()), 'Label di luar 0/1/2!'

imgs_v, labels_v = next(iter(val_loader))
print(f'Val batch   — images: {imgs_v.shape}, labels: {labels_v.tolist()}')

print('\nFold loader LOLOS sanity check!')

In [ ]:
# Visualisasi batch — pastikan gambar tampil wajar
IMAGENET_MEAN_T = torch.tensor(IMAGENET_MEAN).view(3, 1, 1)
IMAGENET_STD_T  = torch.tensor(IMAGENET_STD).view(3, 1, 1)

def denormalize(img_tensor):
    return (img_tensor * IMAGENET_STD_T + IMAGENET_MEAN_T).clamp(0, 1)

fig, axes = plt.subplots(2, 8, figsize=(16, 5))
label_names = ['Recyclable', 'Electronic', 'Organic']

for i in range(8):
    img_show = denormalize(imgs[i]).permute(1, 2, 0).numpy()
    axes[0][i].imshow(img_show)
    axes[0][i].set_title(label_names[labels[i].item()], fontsize=7)
    axes[0][i].axis('off')

    img_show_v = denormalize(imgs_v[i]).permute(1, 2, 0).numpy()
    axes[1][i].imshow(img_show_v)
    axes[1][i].set_title(label_names[labels_v[i].item()], fontsize=7)
    axes[1][i].axis('off')

axes[0][0].set_ylabel('Train (aug)', fontsize=10)
axes[1][0].set_ylabel('Val (no aug)', fontsize=10)
plt.suptitle('Sanity Check: Visualisasi Batch', fontweight='bold')
plt.tight_layout()
plt.savefig(OUTPUT_DIR / 'batch_visualization.png', dpi=100, bbox_inches='tight')
plt.show()

In [ ]:
# Test loader untuk set test — urutan harus sesuai submission.csv
sub_df = pd.read_csv(SUB_CSV)
id_col = sub_df.columns[0]
ordered_filenames = sub_df[id_col].astype(str).tolist()

filepaths = []
for fname in ordered_filenames:
    fp = TEST_DIR / fname
    if not fp.exists():
        for ext in ['.jpg', '.jpeg', '.png']:
            fp_try = TEST_DIR / (fname + ext)
            if fp_try.exists():
                fp = fp_try
                break
    filepaths.append(str(fp))

df_test = pd.DataFrame({'filepath': filepaths})
test_ds = WasteDataset(df_test, transform=get_eval_transform(IMG_SIZE), is_test=True)
test_loader = DataLoader(test_ds, batch_size=16, shuffle=False, num_workers=2)

# Verifikasi 1 batch
imgs_t, fps_t = next(iter(test_loader))
print(f'Test batch — images: {imgs_t.shape}')
print(f'Contoh filepath: {fps_t[0]}')
print(f'Total test gambar: {len(test_ds)} (harapkan 1458)')
assert len(test_ds) == 1458, f'Jumlah test salah: {len(test_ds)}'
print('✅ Test loader LOLOS! Urutan sesuai submission.csv')

## 5. Freeze & Handoff

Setelah semua assertion lolos, folder `OUTPUT_DIR` berisi:
- `folds.csv` → Track B & C
- `class_weights.npy` → Track B
- `eda_stats.json` → Report
- `split_metadata.json` → Dokumentasi

**Upload ke Google Drive / Kaggle Dataset privat untuk sharing.**

In [ ]:
# Final check semua file ada
required_files = ['folds.csv', 'class_weights.npy', 'eda_stats.json', 'split_metadata.json']
all_ok = True
for fname in required_files:
    fp = OUTPUT_DIR / fname
    status = '✅' if fp.exists() else '❌'
    size = fp.stat().st_size if fp.exists() else 0
    print(f'{status} {fname} ({size:,} bytes)')
    if not fp.exists():
        all_ok = False

if all_ok:
    print('\n🎉 Semua artefak siap! Track A DONE — siap handoff ke Track B & C.')
else:
    print('\n⚠️  Ada file yang belum ada. Cek cell di atas.')

## Checklist Fase 2

- [ ] `folds.csv` dibuat (stratified 5-fold, seed dicatat)
- [ ] Group-aware split jika ada duplikat
- [ ] Proporsi kelas per fold & no-overlap terverifikasi
- [ ] `Dataset` + `DataLoader` bisa iterasi 1 batch tanpa error
- [ ] Transform train vs eval + normalisasi ImageNet
- [ ] `class_weights.npy` dihitung
- [ ] Assert mapping label 0/1/2 lolos + batch tervisualisasi
- [ ] Test loader terurut 1..1458
- [ ] Artefak di-freeze & di-share ke Track B & C